#1. Carregar Dados da Silver e Criar Views Temporárias

In [0]:
# Lendo as tabelas da Silver
df_ouvidoria       = spark.read.table("workspace.silver.ouvidoria")
df_usuario         = spark.read.table("workspace.silver.usuario")
df_cidade          = spark.read.table("workspace.silver.cidade")
df_estado          = spark.read.table("workspace.silver.estado")
df_tipo_ouvidoria  = spark.read.table("workspace.silver.tipo_ouvidoria")
df_servico_afetado = spark.read.table("workspace.silver.servico_afetado")

# Criando Views Temporárias para usar no SQL
df_ouvidoria.createOrReplaceTempView("silver_ouvidoria")
df_usuario.createOrReplaceTempView("silver_usuario")
df_cidade.createOrReplaceTempView("silver_cidade")
df_estado.createOrReplaceTempView("silver_estado")
df_tipo_ouvidoria.createOrReplaceTempView("silver_tipo_ouvidoria")
df_servico_afetado.createOrReplaceTempView("silver_servico_afetado")

#2. Dimensão Tempo (Calendário)

In [0]:
from pyspark.sql.functions import expr

data_inicial, data_final = "2023-01-01", "2026-12-31"
num_dias = spark.sql(f"SELECT datediff('{data_final}', '{data_inicial}')").collect()[0][0]

df_tempo = spark.range(0, num_dias + 1) \
    .selectExpr(f"date_add(to_date('{data_inicial}'), CAST(id AS INT)) AS DATA") \
    .selectExpr(
        "DATA", "year(DATA) AS ANO", "month(DATA) AS MES",
        "(CASE month(DATA) WHEN 1 THEN 'JANEIRO' WHEN 2 THEN 'FEVEREIRO' WHEN 3 THEN 'MARCO' WHEN 4 THEN 'ABRIL' WHEN 5 THEN 'MAIO' WHEN 6 THEN 'JUNHO' WHEN 7 THEN 'JULHO' WHEN 8 THEN 'AGOSTO' WHEN 9 THEN 'SETEMBRO' WHEN 10 THEN 'OUTUBRO' WHEN 11 THEN 'NOVEMBRO' WHEN 12 THEN 'DEZEMBRO' END) AS NOME_MES",
        "day(DATA) AS DIA"
    )

df_tempo.write.mode('overwrite').saveAsTable("workspace.gold.dim_tempo")

#3. Dimensão Localidade (Cidades e Estados)

In [0]:
%sql
-- 1. Apaga a tabela antiga caso tenha sido criada com as colunas erradas
DROP TABLE IF EXISTS workspace.gold.dim_localidade;

-- 2. Cria a tabela com os nomes corretos
CREATE TABLE workspace.gold.dim_localidade (
  SK_LOCALIDADE    BIGINT GENERATED BY DEFAULT AS IDENTITY,
  ID_CIDADE        INT,
  NOME_CIDADE      VARCHAR(100),
  NOME_ESTADO      VARCHAR(100),
  SIGLA_ESTADO     VARCHAR(2)
) USING DELTA;

-- 3. Faz o Merge conectando as colunas exatas da sua modelagem
WITH localidade_relacional AS (
    SELECT c.ID_CIDADE, c.NOME_CIDADE, e.NOME_ESTADO, e.SIGLA_ESTADO
    FROM silver_cidade c
    INNER JOIN silver_estado e ON c.COD_ESTADO = e.ID_ESTADO
)
MERGE INTO workspace.gold.dim_localidade AS d
USING localidade_relacional AS r
ON d.ID_CIDADE = r.ID_CIDADE
WHEN MATCHED AND (d.NOME_CIDADE <> r.NOME_CIDADE OR d.NOME_ESTADO <> r.NOME_ESTADO) THEN
  UPDATE SET d.NOME_CIDADE = r.NOME_CIDADE, d.NOME_ESTADO = r.NOME_ESTADO, d.SIGLA_ESTADO = r.SIGLA_ESTADO
WHEN NOT MATCHED THEN
  INSERT (ID_CIDADE, NOME_CIDADE, NOME_ESTADO, SIGLA_ESTADO)
  VALUES (r.ID_CIDADE, r.NOME_CIDADE, r.NOME_ESTADO, r.SIGLA_ESTADO);

#4. Dimensão Usuário

In [0]:
%sql
-- 1. Apaga a tabela antiga caso tenha criado com erro
DROP TABLE IF EXISTS workspace.gold.dim_usuario;

-- 2. Cria a tabela (agora usando CPF_USUARIO)
CREATE TABLE workspace.gold.dim_usuario (
  SK_USUARIO      BIGINT GENERATED BY DEFAULT AS IDENTITY,
  ID_USUARIO      INT,
  NOME_USUARIO    VARCHAR(100),
  CPF_USUARIO     VARCHAR(14)
) USING DELTA;

-- 3. Faz o Merge usando os nomes exatos das colunas da Silver
MERGE INTO workspace.gold.dim_usuario AS d
USING silver_usuario AS r
ON d.ID_USUARIO = r.ID_USUARIO
WHEN MATCHED AND (d.NOME_USUARIO <> r.NOME_USUARIO OR d.CPF_USUARIO <> r.CPF_USUARIO) THEN
  UPDATE SET d.NOME_USUARIO = r.NOME_USUARIO, d.CPF_USUARIO = r.CPF_USUARIO
WHEN NOT MATCHED THEN
  INSERT (ID_USUARIO, NOME_USUARIO, CPF_USUARIO)
  VALUES (r.ID_USUARIO, r.NOME_USUARIO, r.CPF_USUARIO);

#5. Tabela Fato Ouvidoria

In [0]:
%sql
-- 1. Apaga a tabela caso tenha dado erro a meio da criação
DROP TABLE IF EXISTS workspace.gold.fato_ouvidoria;

-- 2. Cria a tabela Fato
CREATE TABLE workspace.gold.fato_ouvidoria (
  FK_TEMPO             DATE,
  FK_USUARIO           BIGINT,
  FK_LOCALIDADE        BIGINT,
  QUANTIDADE_OUVIDORIA INT
) USING DELTA;

-- 3. Insere os dados cruzando com as chaves corretas
INSERT INTO workspace.gold.fato_ouvidoria
SELECT 
    t.DATA             AS FK_TEMPO,
    u_dim.SK_USUARIO   AS FK_USUARIO,
    l.SK_LOCALIDADE    AS FK_LOCALIDADE,
    COUNT(1)           AS QUANTIDADE_OUVIDORIA
FROM silver_ouvidoria o

-- Acha a data
INNER JOIN workspace.gold.dim_tempo t      
    ON CAST(o.DATA_OUVIDORIA AS DATE) = t.DATA

-- Acha a Chave do Usuário na Dimensão Gold
INNER JOIN workspace.gold.dim_usuario u_dim    
    ON o.COD_USUARIO = u_dim.ID_USUARIO

-- A PONTE: Puxa o usuário da Silver só para descobrirmos de que cidade ele é
INNER JOIN silver_usuario u_silv
    ON o.COD_USUARIO = u_silv.ID_USUARIO

-- Liga a cidade do usuário à Dimensão Localidade!
INNER JOIN workspace.gold.dim_localidade l 
    ON u_silv.COD_CIDADE = l.ID_CIDADE

GROUP BY t.DATA, u_dim.SK_USUARIO, l.SK_LOCALIDADE;

-- 4. Mostra o resultado final
SELECT * FROM workspace.gold.fato_ouvidoria LIMIT 10;